# Etapa 1 — Comparación cuantitativa de modelos HAI22.04
Sin entrenamiento, sin recalibración y sin modificación de archivos originales. Compara FULL de DT, RF, ET, XGB y persistencia sobre TEST2–TEST4. Las métricas RMSE/MAE se presentan por señal, sin promediarlas entre escalas distintas.

In [1]:
from pathlib import Path
import re, gzip, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path(r"C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles")
OUT = ROOT / 'analisis' / 'etapa_1'
assert ROOT.is_dir(), f'No existe la carpeta: {ROOT}'
OUT.mkdir(parents=True, exist_ok=True)
TESTS = ('test2','test3','test4')
MODELS = ('DT','RF','ET','XGB','PERSIST')
print('Origen:', ROOT, '\nSalida:', OUT)
for sub in ('metricas','deteccion','predicciones'):
    print(sub, len(list((ROOT/sub).rglob('*'))) if (ROOT/sub).exists() else 'NO EXISTE')

Origen: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles 
Salida: C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles\analisis\etapa_1
metricas 84
deteccion 210
predicciones 189


## 1. Inventario dirigido de archivos y esquemas
Se leen encabezados únicamente. Si una tabla de detección o episodios usa otra convención, el notebook no inventará sus campos: conservará sus datos y reportará qué no pudo consolidar.

In [2]:
def files_in(folder):
    p=ROOT/folder
    return sorted(x for x in p.rglob('*') if x.is_file()) if p.exists() else []
def read_csv(p, nrows=None, usecols=None):
    with p.open('rb') as f: magic=f.read(2)
    return pd.read_csv(p, compression='gzip' if magic==b'\x1f\x8b' else 'infer', nrows=nrows, usecols=usecols, low_memory=False)
def norm(s):return re.sub(r'[^a-z0-9]+','_',str(s).lower()).strip('_')
def model_of(s):
    u=str(s).upper()
    if 'PERSIST' in u:return 'PERSIST'
    for m in ('XGBOOST','XGB','EXTRA_TREES','EXTRATREES','ET','RANDOM_FOREST','RANDOMFOREST','RF','DECISION_TREE','DECISIONTREE','DT'):
        if re.search(r'(^|[^A-Z0-9])'+m+r'([^A-Z0-9]|$)',u):return {'XGBOOST':'XGB','EXTRA_TREES':'ET','EXTRATREES':'ET','RANDOM_FOREST':'RF','RANDOMFOREST':'RF','DECISION_TREE':'DT','DECISIONTREE':'DT'}.get(m,m)
    return None
def test_of(s):
    match=re.search(r'test[_ -]?([234])',str(s),re.I)
    return 'test'+match.group(1) if match else None
def is_full(s):return bool(re.search(r'(^|[^a-z0-9])full([^a-z0-9]|$)',str(s),re.I))
rows=[]
for folder in ('metricas','deteccion','predicciones'):
    for p in files_in(folder):
        if not (p.name.lower().endswith(('.csv','.csv.gz','.gz'))):continue
        try:
            h=read_csv(p,nrows=0)
            rows.append(dict(carpeta=folder,archivo=str(p.relative_to(ROOT)),columnas=' | '.join(h.columns),modelo=model_of(p.name),test=test_of(p.name),full=is_full(p.name),error=''))
        except Exception as e:rows.append(dict(carpeta=folder,archivo=str(p.relative_to(ROOT)),columnas='',modelo=model_of(p.name),test=test_of(p.name),full=is_full(p.name),error=str(e)))
inventory=pd.DataFrame(rows)
inventory.to_csv(OUT/'01_inventario_dirigido.csv',index=False,encoding='utf-8-sig')
display(inventory[inventory.carpeta.isin(['metricas','deteccion'])].head(30))
print('Archivos:',len(inventory),'Errores encabezado:',int(inventory.error.ne('').sum()))

,carpeta,archivo,columnas,modelo,test,full,error
0,metricas,metricas\comparacion_deteccion.csv,metodo | config | archivo | n | n_alerta | fra...,None,None,False,
1,metricas,metricas\deteccion_DT_FULL.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,True,
2,metricas,metricas\deteccion_DT_SIN_P1.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
3,metricas,metricas\deteccion_DT_SIN_P2.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
4,metricas,metricas\deteccion_DT_SIN_P3.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
5,metricas,metricas\deteccion_DT_SIN_P4.csv,metodo | config | archivo | n | n_alerta | fra...,DT,None,False,
6,metricas,metricas\deteccion_ET_FULL.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,True,
7,metricas,metricas\deteccion_ET_SIN_P1.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,False,
8,metricas,metricas\deteccion_ET_SIN_P2.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,False,
9,metricas,metricas\deteccion_ET_SIN_P3.csv,metodo | config | archivo | n | n_alerta | fra...,ET,None,False,


Archivos: 462 Errores encabezado: 0


## 2. Métricas predictivas por señal y TEST
Se conserva RMSE y MAE en unidades originales. Los promedios de métricas por señal de diferentes escalas no se calculan.

In [3]:
predict=[]; issues=[]
for p in files_in('metricas'):
    if not re.search(r'^prediccion_.*\.csv(?:\.gz)?$',p.name,re.I):continue
    try:
        d=read_csv(p); d.columns=[norm(x) for x in d.columns]
        required={'variable','componente','rmse','mae'}
        if not required.issubset(d.columns):issues.append(f'{p.name}: faltan {required-set(d.columns)}');continue
        d['modelo']=d['metodo'].map(model_of) if 'metodo' in d else model_of(p.name)
        if 'configuracion' in d:
            d=d[d['configuracion'].astype(str).map(is_full)]
        elif not is_full(p.name):continue
        d['test']=d['archivo'].map(test_of) if 'archivo' in d else test_of(p.name)
        d=d[d['modelo'].isin(MODELS)&d['test'].isin(TESTS)].copy()
        d['fuente']=str(p.relative_to(ROOT))
        predict.append(d[['modelo','test','componente','variable','rmse','mae','fuente']])
    except Exception as e:issues.append(f'{p.name}: {e}')
metrics=pd.concat(predict,ignore_index=True) if predict else pd.DataFrame(columns=['modelo','test','componente','variable','rmse','mae','fuente'])
metrics['rmse']=pd.to_numeric(metrics.rmse,errors='coerce');metrics['mae']=pd.to_numeric(metrics.mae,errors='coerce')
metrics.to_csv(OUT/'02_metricas_por_senal.csv',index=False,encoding='utf-8-sig')
print('Filas:',len(metrics),'Modelos:',sorted(metrics.modelo.unique()),'TEST:',sorted(metrics.test.unique()))
print('Incidencias:',issues[:20]);display(metrics.head())

Filas: 360 Modelos: ['DT', 'ET', 'PERSIST', 'RF', 'XGB'] TEST: ['test2', 'test3', 'test4']
Incidencias: []


,modelo,test,componente,variable,rmse,mae,fuente
0,DT,test2,P1,P1_FT01,7.717825,5.726655,metricas\prediccion_DT_FULL.csv
1,DT,test2,P1,P1_FT02,188.412827,79.054329,metricas\prediccion_DT_FULL.csv
2,DT,test2,P1,P1_FT03,19.391403,14.962177,metricas\prediccion_DT_FULL.csv
3,DT,test2,P1,P1_LIT01,31.037125,22.510300,metricas\prediccion_DT_FULL.csv
4,DT,test2,P1,P1_PIT01,0.083799,0.060072,metricas\prediccion_DT_FULL.csv


## 3. Comparaciones predictivas pareadas frente a persistencia
Cada diferencia se calcula sobre la misma señal y TEST. Valores negativos de ΔRMSE o ΔMAE indican mejora respecto a persistencia. Se cuentan señales que mejoran/empeoran sin agregar sus magnitudes heterogéneas.

In [4]:
base=metrics[metrics.modelo=='PERSIST'][['test','componente','variable','rmse','mae']].rename(columns={'rmse':'rmse_persist','mae':'mae_persist'})
paired=metrics[metrics.modelo!='PERSIST'].merge(base,on=['test','componente','variable'],how='inner',validate='many_to_one') if not base.duplicated(['test','componente','variable']).any() else pd.DataFrame()
if len(paired):
    paired['delta_rmse']=paired.rmse-paired.rmse_persist
    paired['delta_mae']=paired.mae-paired.mae_persist
    paired['mejora_rmse_pct']=np.where(paired.rmse_persist>0,100*(paired.rmse_persist-paired.rmse)/paired.rmse_persist,np.nan)
    paired.to_csv(OUT/'03_comparacion_pareada_por_senal.csv',index=False,encoding='utf-8-sig')
    summary=paired.groupby(['modelo','test','componente'],dropna=False).agg(senales=('variable','nunique'),mejor_rmse=('delta_rmse',lambda x:int((x<0).sum())),peor_rmse=('delta_rmse',lambda x:int((x>0).sum())),igual_rmse=('delta_rmse',lambda x:int((x==0).sum())),mejor_mae=('delta_mae',lambda x:int((x<0).sum()))).reset_index()
    summary.to_csv(OUT/'04_balance_predictivo.csv',index=False,encoding='utf-8-sig')
    display(summary)
else: print('No hay pares comparables. Revisar 02_metricas_por_senal.csv y los nombres de columnas.')

,modelo,test,componente,senales,mejor_rmse,peor_rmse,igual_rmse,mejor_mae
0,DT,test2,P1,9,0,9,0,0
1,DT,test2,P2,6,0,6,0,0
2,DT,test2,P3,3,0,3,0,0
3,DT,test2,P4,6,2,4,0,1
4,DT,test3,P1,9,0,9,0,0
5,DT,test3,P2,6,0,6,0,0
6,DT,test3,P3,3,0,3,0,0
7,DT,test3,P4,6,2,4,0,1
8,DT,test4,P1,9,0,9,0,0
9,DT,test4,P2,6,0,6,0,0


## 4. Detección y episodios: lectura de las fuentes correctas
Los agregados de detección y episodios se encuentran en `metricas/`, mientras las alertas por registro están en `deteccion/`. Se filtran las filas de TEST2–TEST4 dentro de cada CSV y solo la configuración FULL. Se procesan archivos individualmente, sin crear una tabla de millones de filas.

In [5]:
def classify(p):
    s=p.name.lower()
    return 'episodios' if 'episodios_' in s else 'resumen_deteccion' if 'deteccion_' in s else 'alertas_registro'

def canon_model(value):
    x=str(value).strip().upper()
    return model_of(x) or (x if x in MODELS else None)

def canon_test(value):
    return test_of(value)

def full_rows(d, p):
    d=d.copy(); d.columns=[norm(c) for c in d.columns]
    if 'config' in d: conf=d['config'].astype(str).str.upper().eq('FULL')
    elif 'configuracion' in d: conf=d['configuracion'].astype(str).str.upper().eq('FULL')
    else: conf=pd.Series(is_full(p.name), index=d.index)
    return d.loc[conf].copy()

index=[]
for folder in ('metricas','deteccion'):
    for p in files_in(folder):
        if not p.name.lower().endswith(('.csv','.csv.gz')): continue
        if folder=='metricas' and not re.match(r'^(deteccion|episodios)_',p.name,re.I):continue
        if folder=='deteccion' and not (is_full(p.name) and test_of(p.name) in TESTS):continue
        try:
            header=read_csv(p,nrows=0)
            index.append(dict(archivo=str(p.relative_to(ROOT)),tipo=classify(p) if folder=='metricas' else 'alertas_registro',modelo=model_of(p.name),test=test_of(p.name),full=is_full(p.name),columnas=' | '.join(map(norm,header.columns)),error=''))
        except Exception as e: index.append(dict(archivo=str(p.relative_to(ROOT)),tipo='error',modelo=model_of(p.name),test=test_of(p.name),full=is_full(p.name),columnas='',error=str(e)))
idx=pd.DataFrame(index)
idx.to_csv(OUT/'05_inventario_deteccion.csv',index=False,encoding='utf-8-sig')
print('Fuentes de detección/episodios:',len(idx))
display(idx.groupby('tipo').size().rename('archivos').reset_index())

Fuentes de detección/episodios: 57


,tipo,archivos
0,alertas_registro,15
1,episodios,21
2,resumen_deteccion,21


## 5. Métricas de detección calculadas por TEST
La tabla `metricas/deteccion_*_FULL.csv` contiene contadores por TEST y `metricas/episodios_*_FULL.csv` contiene un registro por episodio. Se contrastan los contadores de alertas con `deteccion/*_FULL_test*.csv.gz`; nunca se infieren episodios desde cantidad de alertas.

In [6]:
records=[]; episode_rows=[]; warnings=[]
for p in files_in('metricas'):
    if not re.match(r'^(deteccion|episodios)_.+_FULL\.csv(?:\.gz)?$',p.name,re.I):continue
    method=model_of(p.name)
    if method not in MODELS:continue
    try:
        d=full_rows(read_csv(p),p)
        if not {'archivo'}.issubset(d.columns):
            warnings.append(f'{p.name}: no existe columna archivo');continue
        d['test_etapa1']=d['archivo'].map(canon_test)
        d=d[d.test_etapa1.isin(TESTS)]
        if p.name.lower().startswith('episodios_'):
            needed={'inicio_indice','fin_indice','detectado'}
            if not needed.issubset(d.columns):warnings.append(f'{p.name}: faltan {needed-set(d.columns)}');continue
            d['detectado_num']=pd.to_numeric(d.detectado,errors='coerce')
            if d.detectado_num.isna().any():
                warnings.append(f'{p.name}: detectado con valores no numéricos');continue
            for t,g in d.groupby('test_etapa1'):
                episode_rows.append(dict(modelo=method,test=t,episodios_totales=len(g),episodios_detectados=int((g.detectado_num==1).sum()),episodios_no_detectados=int((g.detectado_num==0).sum()),fuente=str(p.relative_to(ROOT))))
        else:
            required={'n','n_alerta'}
            if not required.issubset(d.columns):warnings.append(f'{p.name}: faltan {required-set(d.columns)}');continue
            for _,r in d.iterrows():
                t=r.test_etapa1
                for src,dst in [('n','registros'),('n_alerta','alertas_totales'),('fraccion_alerta','fraccion_alerta'),('episodios_visibles','episodios_visibles_resumen'),('alertas_fuera_etiqueta','alertas_fuera_etiqueta_resumen'),('episodios_detectados','episodios_detectados_resumen')]:
                    if src in d.columns and pd.notna(r[src]):
                        records.append(dict(modelo=method,test=t,metrica=dst,valor=float(r[src]),fuente=str(p.relative_to(ROOT))))
    except Exception as e:warnings.append(f'{p.name}: {e}')

# El conteo por registro es una comprobación independiente: se lee cada archivo una sola vez.
alert_rows=[]
for p in files_in('deteccion'):
    m=model_of(p.name);t=test_of(p.name)
    if m not in MODELS or t not in TESTS or not is_full(p.name):continue
    try:
        d=read_csv(p,usecols=lambda c: norm(c) in ('alerta','attack'))
        d.columns=[norm(c) for c in d.columns]
        if not {'alerta','attack'}.issubset(d.columns):warnings.append(f'{p.name}: faltan alerta/attack');continue
        a=pd.to_numeric(d.alerta,errors='coerce');attack=pd.to_numeric(d.attack,errors='coerce')
        if a.isna().any() or attack.isna().any():warnings.append(f'{p.name}: alerta/attack no numérico');continue
        al=a.eq(1);atk=attack.eq(1)
        alert_rows.append(dict(modelo=m,test=t,registros_observados=len(d),alertas_observadas=int(al.sum()),alertas_en_ataque=int((al&atk).sum()),alertas_fuera_etiqueta=int((al&~atk).sum()),fuente=str(p.relative_to(ROOT))))
    except Exception as e:warnings.append(f'{p.name}: {e}')

summary=pd.DataFrame(records,columns=['modelo','test','metrica','valor','fuente'])
summary.to_csv(OUT/'07_metricas_deteccion_identificadas.csv',index=False,encoding='utf-8-sig')
episodes=pd.DataFrame(episode_rows,columns=['modelo','test','episodios_totales','episodios_detectados','episodios_no_detectados','fuente'])
episodes.to_csv(OUT/'08_episodios_por_test.csv',index=False,encoding='utf-8-sig')
alerts=pd.DataFrame(alert_rows,columns=['modelo','test','registros_observados','alertas_observadas','alertas_en_ataque','alertas_fuera_etiqueta','fuente'])
alerts.to_csv(OUT/'08_alertas_por_test.csv',index=False,encoding='utf-8-sig')

wide=(summary.pivot_table(index=['modelo','test'],columns='metrica',values='valor',aggfunc='first').reset_index() if len(summary) else pd.DataFrame(columns=['modelo','test']))
if len(episodes):wide=wide.merge(episodes.drop(columns='fuente'),on=['modelo','test'],how='outer',validate='one_to_one')
if len(alerts):wide=wide.merge(alerts.drop(columns='fuente'),on=['modelo','test'],how='outer',validate='one_to_one')
if 'alertas_totales' in wide and 'alertas_observadas' in wide:
    mismatch=wide[wide.alertas_totales.notna() & wide.alertas_observadas.notna() & (wide.alertas_totales!=wide.alertas_observadas)]
    if len(mismatch):warnings.append('Diferencias entre n_alerta y alertas observadas: '+str(mismatch[['modelo','test']].values.tolist()))
if 'episodios_detectados_resumen' in wide and 'episodios_detectados' in wide:
    mismatch=wide[wide.episodios_detectados_resumen.notna() & wide.episodios_detectados.notna() & (wide.episodios_detectados_resumen!=wide.episodios_detectados)]
    if len(mismatch):warnings.append('Diferencias en episodios detectados: '+str(mismatch[['modelo','test']].values.tolist()))
wide.to_csv(OUT/'09_tabla_deteccion_por_modelo_y_test.csv',index=False,encoding='utf-8-sig')
for col in ('episodios_detectados','alertas_observadas','alertas_fuera_etiqueta'):
    if col not in wide or wide[col].isna().all():continue
    pv=wide.pivot(index='modelo',columns='test',values=col).reindex(index=MODELS,columns=TESTS)
    ax=pv.plot(kind='bar',figsize=(9,4.5),rot=0)
    ax.set(title=col.replace('_',' ').capitalize(),ylabel='Cantidad',xlabel='Modelo')
    ax.grid(axis='y',alpha=.25)
    plt.tight_layout();plt.savefig(OUT/f'09_{col}.png',dpi=180);plt.close()
print('Episodios:',len(episodes),'filas; alertas:',len(alerts),'filas; resumen:',len(summary),'métricas')
display(wide)
if warnings:print('ADVERTENCIAS:',*warnings,sep='\n - ')

Episodios: 15 filas; alertas: 15 filas; resumen: 90 métricas


,modelo,test,alertas_fuera_etiqueta_resumen,alertas_totales,episodios_detectados_resumen,episodios_visibles_resumen,fraccion_alerta,registros,episodios_totales,episodios_detectados,episodios_no_detectados,registros_observados,alertas_observadas,alertas_en_ataque,alertas_fuera_etiqueta
0,DT,test2,833.0,998.0,10.0,17.0,0.012058,82770.0,17,10,7,82770,998,165,833
1,DT,test3,440.0,1048.0,7.0,10.0,0.016803,62370.0,10,7,3,62370,1048,608,440
2,DT,test4,22190.0,23217.0,24.0,24.0,0.179185,129570.0,24,24,0,129570,23217,1027,22190
3,ET,test2,12798.0,13545.0,13.0,17.0,0.163646,82770.0,17,13,4,82770,13545,747,12798
4,ET,test3,8009.0,9474.0,8.0,10.0,0.151900,62370.0,10,8,2,62370,9474,1465,8009
5,ET,test4,44801.0,46739.0,24.0,24.0,0.360724,129570.0,24,24,0,129570,46739,1938,44801
6,PERSIST,test2,500.0,598.0,13.0,17.0,0.007225,82770.0,17,13,4,82770,598,98,500
7,PERSIST,test3,391.0,748.0,9.0,10.0,0.011993,62370.0,10,9,1,62370,748,357,391
8,PERSIST,test4,766.0,981.0,23.0,24.0,0.007571,129570.0,24,23,1,129570,981,215,766
9,RF,test2,6911.0,7311.0,13.0,17.0,0.088329,82770.0,17,13,4,82770,7311,400,6911


## 6. Figuras predictivas por señal y TEST
Se genera una figura por componente y métrica; no se promedian escalas.

In [7]:
for comp in sorted(metrics.componente.dropna().unique()):
    part=metrics[metrics.componente==comp]
    for metric in ('rmse','mae'):
        for test in TESTS:
            d=part[part.test==test]
            if d.empty:continue
            pivot=d.pivot_table(index='variable',columns='modelo',values=metric,aggfunc='first')
            if pivot.empty:continue
            ax=pivot.plot(kind='bar',figsize=(max(8,len(pivot)*.65),5),rot=65)
            ax.set_title(f'{metric.upper()} por señal — {comp} — {test.upper()}')
            ax.set_ylabel(f'{metric.upper()} (unidad original de cada señal)');ax.set_xlabel('Señal')
            ax.grid(axis='y',alpha=.2);plt.tight_layout()
            safe=re.sub('[^a-zA-Z0-9_-]','_',str(comp))
            plt.savefig(OUT/f'09_{metric}_{safe}_{test}.png',dpi=180,bbox_inches='tight');plt.close()
print('Figuras predictivas generadas:',len(list(OUT.glob('09_*.png'))))

Figuras predictivas generadas: 27


## 7. Verificación final y reporte de cobertura
Los archivos no encontrados o campos no interpretables quedan documentados. No se declara completada la comparación de detección si faltan sus métricas.

In [8]:
expected={(m,t) for m in MODELS for t in TESTS}
actual=set(zip(metrics.modelo,metrics.test))
with_episode=set(zip(episodes.modelo,episodes.test))
with_alert=set(zip(alerts.modelo,alerts.test))
with_summary=set(zip(summary.modelo,summary.test))
coverage=pd.DataFrame([dict(modelo=m,test=t,metricas_predictivas=(m,t) in actual,episodios=(m,t) in with_episode,alertas=(m,t) in with_alert,resumen_deteccion=(m,t) in with_summary,metricas_deteccion=((m,t) in with_episode and (m,t) in with_alert and (m,t) in with_summary)) for m,t in sorted(expected)])
coverage.to_csv(OUT/'10_cobertura_etapa1.csv',index=False,encoding='utf-8-sig')
(OUT/'11_incidencias.txt').write_text('\n'.join(issues+warnings) if issues or warnings else 'Sin incidencias',encoding='utf-8')
required=['02_metricas_por_senal.csv','07_metricas_deteccion_identificadas.csv','08_episodios_por_test.csv','08_alertas_por_test.csv','09_tabla_deteccion_por_modelo_y_test.csv','10_cobertura_etapa1.csv']
for f in required:assert (OUT/f).is_file(),f'No se generó: {f}'
print('Cobertura completa detección:',int(coverage.metricas_deteccion.sum()),'de',len(coverage))
display(coverage)
assert coverage.metricas_deteccion.all(), 'Faltan métricas de detección de alguna combinación FULL/TEST: revisar 11_incidencias.txt' 

Cobertura completa detección: 15 de 15


,modelo,test,metricas_predictivas,episodios,alertas,resumen_deteccion,metricas_deteccion
0,DT,test2,True,True,True,True,True
1,DT,test3,True,True,True,True,True
2,DT,test4,True,True,True,True,True
3,ET,test2,True,True,True,True,True
4,ET,test3,True,True,True,True,True
5,ET,test4,True,True,True,True,True
6,PERSIST,test2,True,True,True,True,True
7,PERSIST,test3,True,True,True,True,True
8,PERSIST,test4,True,True,True,True,True
9,RF,test2,True,True,True,True,True
